Updated inference notebook

Run from top to bottom in your existing Darts environment and modelling directory.
Uses your existing September 15 best checkpoint and cached histories.

Changes: corrected PyTorch negative-binomial mean and SciPy quantiles; deterministic
installed-likelihood consistency check; fresh output directory per run; date,
parameter and output checks. Previous execution outputs have been cleared.

Full model execution requires your checkpoint, column_roles.json and series_cache.
Monthly sums of individual quantiles are not quantiles of total demand.


In [ ]:
import pickle
import os
import json
import glob
import gc
import collections.abc
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from darts import TimeSeries
from darts.dataprocessing.transformers import StaticCovariatesTransformer
from darts.models import TFTModel
from darts.utils.likelihood_models import (
    NegativeBinomialLikelihood,
    PoissonLikelihood,
)
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Use the existing best checkpoint; no retraining is required.
MODEL_NAME = 'daily_tft_negbin_scooters_2026-09-15_20_02_52'


In [ ]:
#Loading the CACHE_DIR
BASE = os.getcwd()
CACHE_DIR       = os.path.join(os.getcwd(), "series_cache")


#Loading the ROLES Json : This helps to avoid hardcoding the variables
with open(os.path.join(BASE,"column_roles.json")) as f:
    ROLES = json.load(f)

time_col,group_col,target_col = ROLES["time_col"],ROLES["group_col"],ROLES["target_col"]

FREQ = ROLES["freq"]
static_covariates = ROLES["static_covariates"]

FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END = pd.Timestamp(ROLES["forecast_end"])
HORIZON = (FORECAST_END - FORECAST_START).days + 1

safe_name = lambda k: str(k).replace("<>", "_").replace("/", "_").replace("\\", "_")

In [ ]:
# ---------------- SEQUENCES ----------------
class History(collections.abc.Sequence):
    """Series history to forecast from. Reads the cached 'val' split,
    which ends on the day before FORECAST_START."""

    def __init__(self, keys, statics):
        self.keys, self.statics = keys, statics

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
            sales, start = z["val_sales"], str(z["val_start"])
        times = pd.date_range(start, periods=len(sales), freq=FREQ)
        if times[-1] != FORECAST_START - pd.Timedelta(days=1):
            raise ValueError(f"History for {self.keys[i]} ends on {times[-1]}.")
        return TimeSeries.from_times_and_values(
            times,
            sales.reshape(-1, 1).astype(np.float32),
            columns=[target_col],
            static_covariates=self.statics[i],
        )


class SharedCov(collections.abc.Sequence):
    def __init__(self, cov, n):
        self.cov, self.n = cov, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.cov for _ in range(*i.indices(self.n))]
        return self.cov

In [ ]:
print(f"Forecast: {FORECAST_START.date()} -> {FORECAST_END.date()} ({HORIZON} days)")

In [ ]:
BATCH_SIZE = 512        # raise until GPU memory complains
BLOCK      = 5000       # series per block, written to disk as it goes
LIMIT      = None       # set to e.g. 2000 for a quick smoke test

In [ ]:
with open(os.path.join(CACHE_DIR, "manifest.json")) as f:
    manifest = json.load(f)

static_df = pd.read_parquet(os.path.join(CACHE_DIR, "static_covariates.parquet"))

# same dead-series filter as training, so indices stay aligned with static_df
keep = []
for k in manifest["series_keys"]:
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
        keep.append(z["train_sales"].sum() > 0)

series_keys = [k for k, m in zip(manifest["series_keys"], keep) if m]
has_val     = [h for h, m in zip(manifest["has_val"],     keep) if m]
static_df   = static_df.loc[keep].reset_index(drop=True)[static_covariates].astype(str)

keys = [k for k, h in zip(series_keys, has_val) if h]
idxs = [i for i, h in enumerate(has_val) if h]
if LIMIT:
    keys, idxs = keys[:LIMIT], idxs[:LIMIT]

print(f"Series to forecast: {len(keys):,}")
if len(keys) < len(series_keys):
    print(f"  ({len(series_keys) - len(keys):,} lack enough history and are skipped)")

with open(os.path.join(CACHE_DIR, "static_cov_transformer.pkl"), "rb") as f:
    sc_transformer = pickle.load(f)

SHARED_COV = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
if SHARED_COV.end_time() < FORECAST_END:
    raise ValueError(f"Covariates end {SHARED_COV.end_time().date()}, need {FORECAST_END.date()}")

# encode static covariates once, not once per series inside the loader
dummy_t = pd.date_range("2000-01-01", periods=2, freq="D")
statics = []
for i in idxs:
    t = TimeSeries.from_times_and_values(
        dummy_t, np.zeros((2, 1), dtype=np.float32), columns=[target_col],
        static_covariates=static_df.iloc[[i]].reset_index(drop=True),
    )
    statics.append(sc_transformer.transform(t).static_covariates)

# load BEST weights -- model.predict() on an in-memory model uses the LAST
# epoch, which with patience=5 is 5 epochs past what early stopping picked
model = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)
print(f"Loaded best checkpoint: {MODEL_NAME}")


In [ ]:
# Every execution gets a fresh directory so old decoding results cannot be reused.
from uuid import uuid4
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
OUT_DIR = os.path.join(
    BASE, "predictions_2026_negbin_torch_convention", RUN_ID
)
print("Output directory:", OUT_DIR)


In [ ]:
from scipy import stats

In [ ]:
# ---------------- NEGATIVE-BINOMIAL DECODING ----------------
# Darts 0.40 returns (r, p) passed to torch.distributions.NegativeBinomial.
# PyTorch mean: r*p/(1-p). SciPy nbinom requires probability 1-p.
# Do not interpret Darts' internal variable name "mu" as the reported mean.
QUANTILES = [35, 40, 50, 60]


def summarise(p_ts):
    """Decode the predictive distribution using PyTorch's probability convention."""
    expected = [f"{target_col}_r", f"{target_col}_p"]
    if list(p_ts.components) != expected:
        raise ValueError(f"Expected {expected}, received {list(p_ts.components)}")
    v = p_ts.values(copy=False).astype(np.float64)
    r, pr = v[:, 0], v[:, 1]
    if (not np.isfinite(v).all() or np.any(r <= 0)
            or np.any((pr <= 0) | (pr >= 1))):
        raise ValueError("Invalid negative-binomial parameters; inspect precision/output.")
    # Fail on invalid parameters rather than silently clipping them.
    out = {"PRED_MEAN": r * pr / (1.0 - pr)}
    for q_int in QUANTILES:
        out[f"PRED_Q{q_int}"] = stats.nbinom.ppf(q_int / 100.0, r, 1.0 - pr)
    if not all(np.isfinite(x).all() for x in out.values()):
        raise ValueError("Non-finite decoded predictions.")
    return out


# Check the installed likelihood implementation before forecasting all series.
# Synthetic raw outputs exercise both Darts' parameter export and its training
# distribution. This check is deterministic and does not run the TFT network.
import darts
print("Darts version:", darts.__version__)
if not isinstance(model.likelihood, NegativeBinomialLikelihood):
    raise TypeError("This notebook requires NegativeBinomialLikelihood.")
if not keys or len(set(keys)) != len(keys):
    raise ValueError("Forecast series keys must be non-empty and unique.")
if not 0 < HORIZON <= model.output_chunk_length:
    raise ValueError("Parameter prediction requires horizon <= output_chunk_length.")

_raw = torch.tensor([[[[-2.0, -1.0]], [[0.4, -0.2]], [[3.0, 1.5]]]],
                    dtype=torch.float64)
_lk = model.likelihood
_dist = _lk._distr_from_params(_lk._params_from_output(_raw))
_exported = _lk.predict_likelihood_parameters(_raw).detach().cpu().numpy()
_probe_ts = TimeSeries.from_times_and_values(
    pd.date_range("2000-01-01", periods=3, freq="D"),
    _exported.reshape(3, 2), columns=[f"{target_col}_r", f"{target_col}_p"],
)
_probe_summary = summarise(_probe_ts)
np.testing.assert_allclose(
    _probe_summary["PRED_MEAN"], _dist.mean.detach().cpu().numpy().ravel(),
    rtol=1e-10, atol=1e-10,
)
# Verify the SciPy probability conversion against Torch's probability mass.
_r, _p = _exported.reshape(3, 2).T
_counts = torch.tensor([0.0, 2.0, 5.0], dtype=torch.float64).reshape(1, 3, 1)
np.testing.assert_allclose(
    stats.nbinom.logpmf(_counts.numpy().ravel(), _r, 1.0 - _p),
    _dist.log_prob(_counts).detach().cpu().numpy().ravel(),
    rtol=1e-10, atol=1e-10,
)
print("PASS: decoded means and SciPy probabilities match the installed likelihood.")

# ---------------- FORECAST ----------------
EXPECTED_PARAMS = [f"{target_col}_r", f"{target_col}_p"]
EXPECTED_COLS   = {"PRED_MEAN", *(f"PRED_Q{q}" for q in QUANTILES)}

os.makedirs(OUT_DIR, exist_ok=True)
n_blocks = (len(keys) + BLOCK - 1) // BLOCK
paths, t0 = [], datetime.now()

for b in range(n_blocks):
    lo, hi = b * BLOCK, min((b + 1) * BLOCK, len(keys))
    path = os.path.join(OUT_DIR, f"{MODEL_NAME}_block_{b:04d}.parquet")
    paths.append(path)

    preds = model.predict(
        n=HORIZON,
        series=History(keys[lo:hi], statics[lo:hi]),
        future_covariates=SharedCov(SHARED_COV, hi - lo),
        predict_likelihood_parameters=True,   # 1 pass instead of num_samples passes
        num_samples=1,
        batch_size=BATCH_SIZE,
        verbose=False,
    )

    if len(preds) != hi - lo:
        raise ValueError("Prediction count does not match the requested series.")
    for p_ts in preds:
        if (len(p_ts) != HORIZON or p_ts.start_time() != FORECAST_START
                or p_ts.end_time() != FORECAST_END):
            raise ValueError("Forecast dates do not match the requested horizon.")

    pd.concat(
        [pd.DataFrame({group_col: k, time_col: p_ts.time_index, **summarise(p_ts)})
         for k, p_ts in zip(keys[lo:hi], preds)],
        ignore_index=True,
    ).to_parquet(path, index=False)

    el = (datetime.now() - t0).total_seconds()
    print(f"[{b+1}/{n_blocks}] {hi:,}/{len(keys):,} | {hi/el:,.0f} series/s | "
          f"ETA {(len(keys)-hi)/(hi/el)/60:.0f} min")

    del preds
    gc.collect()


# ---------------- OUTPUT ----------------
df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
out = os.path.join(OUT_DIR, f"{MODEL_NAME}_predictions.parquet")
if len(df) != len(keys) * HORIZON or df.duplicated([group_col, time_col]).any():
    raise ValueError("Unexpected row count or duplicate series/date predictions.")
df.to_parquet(out, index=False)

print(f"\nWritten -> {out}")
print(f"  rows: {len(df):,} | elapsed: {(datetime.now()-t0).total_seconds()/60:.1f} min")
for c in ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]:
    print(f"  {c}: {df[c].sum():,.0f} ({df[c].sum()/1e5:.2f} lacs)")

In [ ]:
# Read the predictions back from the path this run just wrote,
# rather than a hardcoded absolute path to the previous (wrong) output.
parquet_pred = pd.read_parquet(out)
print(f"{len(parquet_pred):,} rows from {out}")

In [ ]:
# Previous output directories are retained for comparison.


In [ ]:
parquet_pred.head()

In [ ]:
parquet_pred[time_col] = pd.to_datetime(parquet_pred[time_col])
parquet_pred["MONTH_NAME"] = parquet_pred[time_col].dt.month_name()

In [ ]:
# Historical shares below are reference estimates from a subset, not ground truth.


In [ ]:
# ---------------- SHAPE CHECK ----------------
# The total being plausible means little on its own -- the failure mode here was
# always the SHAPE. Compare the predicted month split against what the last three
# festive seasons actually did.
#
# The historical shares below are calendar-drift corrected: each past year was
# mapped onto 2026's month boundaries in festive-offset terms (2026 has N=Oct 11,
# D=Nov 8, so Sep = N-40..N-11, Oct = N-10..D-8, Nov = D-7..D+22, Dec = D+23..D+29)
# using the real anchors from Festive_Data_for_Daily_forecasting.csv. Derived from
# the 07_festive_profile.py export, which covers a ~8% subset of series -- so treat
# these as shape guidance, not exact population shares.
HIST_SHARE = {        # % of the Sep 1 - Dec 7 total
    "September": {"2023": 10.9, "2024":  4.8, "2025":  7.4, "mean":  7.7},
    "October":   {"2023": 16.8, "2024": 16.0, "2025": 19.1, "mean": 17.3},
    "November":  {"2023": 71.2, "2024": 78.2, "2025": 71.3, "mean": 73.6},
    "December":  {"2023":  1.1, "2024":  1.0, "2025":  2.1, "mean":  1.4},
}

parquet_pred[time_col]  = pd.to_datetime(parquet_pred[time_col])
parquet_pred["MONTH_NAME"] = parquet_pred[time_col].dt.month_name()

ORDER = ["September", "October", "November", "December"]
COLS  = ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]

split = (parquet_pred.groupby("MONTH_NAME")[COLS].sum()
         .reindex(ORDER))

print("TOTALS BY MONTH")
print(split.round(0).to_string())

print("\nSHARE OF WINDOW (%) -- predicted vs actual history")
share = 100 * split / split.sum().replace(0, np.nan)
share.insert(0, "HIST_mean", [HIST_SHARE[m]["mean"] for m in ORDER])
print(share.round(1).to_string())

gap = share["PRED_MEAN"] - share["HIST_mean"]
print("\nPRED_MEAN share minus historical mean share (pp):")
print(gap.round(1).to_string())
print("\nNote: December is only 7 days (Dec 1-7), so a small share is expected.")

In [ ]:
import os, json
import numpy as np
import pandas as pd
from darts import TimeSeries

In [ ]:
BASE      = os.getcwd()
CACHE_DIR = os.path.join(os.getcwd(), "series_cache")

In [ ]:
FC_START = pd.Timestamp("2026-09-01")
FC_END   = pd.Timestamp("2026-12-07")

with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
time_col = ROLES["time_col"]

N_BLOCK = [f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
D_BLOCK = [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]


def frame_from_pickle(path):
    ts = TimeSeries.from_pickle(path)
    df = ts.to_dataframe().reset_index()
    df.columns = [time_col] + list(ts.components)
    return df


# ------------------------------------------------------------------ load both
sources = {}

pkl = os.path.join(CACHE_DIR, "shared_cov.pkl")
if os.path.exists(pkl):
    sources["shared_cov.pkl (what inference uses)"] = frame_from_pickle(pkl)
else:
    print("shared_cov.pkl NOT FOUND -- inference would have rebuilt from parquet.")

pq = os.path.join(BASE, "shared_calendar.parquet")
if os.path.exists(pq):
    d = pd.read_parquet(pq)
    d[time_col] = pd.to_datetime(d[time_col])
    sources["shared_calendar.parquet (source)"] = d


for label, cal in sources.items():
    print("=" * 70)
    print(label)
    print("=" * 70)

    cal[time_col] = pd.to_datetime(cal[time_col])
    print(f"Date range : {cal[time_col].min().date()} -> {cal[time_col].max().date()}")
    print(f"Columns    : {len(cal.columns) - 1}")

    if cal[time_col].max() < FC_END:
        print(f"\n*** FAIL: calendar ends before {FC_END.date()}. "
              f"Covariates are missing for part of the forecast. ***")

    win = cal[(cal[time_col] >= FC_START) & (cal[time_col] <= FC_END)]
    print(f"Rows in Sep 1 - Dec 7 2026: {len(win)} (expected 98)")

    if len(win) == 0:
        print("\n*** FAIL: no calendar rows in the forecast window at all. ***\n")
        continue

    # --- NaN check: NaNs silently poison the whole forecast ------------------
    nan_cols = [c for c in win.columns if c != time_col and win[c].isna().any()]
    if nan_cols:
        print(f"\n*** FAIL: {len(nan_cols)} columns contain NaN in the window: "
              f"{nan_cols[:8]} ***")
    else:
        print("NaN check : clean")

    # --- the festive blocks --------------------------------------------------
    for name, block, expect in [("N block", N_BLOCK, 27), ("D block", D_BLOCK, 10)]:
        present = [c for c in block if c in win.columns]
        missing = [c for c in block if c not in win.columns]

        print(f"\n{name}: {len(present)}/{len(block)} columns present")
        if missing:
            print(f"  MISSING COLUMNS: {missing}")

        live = {c: int((win[c] != 0).sum()) for c in present}
        n_live = sum(1 for v in live.values() if v > 0)
        print(f"  columns with a non-zero day in the window: {n_live}/{expect}")

        if n_live == 0:
            print("  *** FAIL: the entire block is zero across the forecast window. ***")
            print("      The model is flying blind on this festival. This is the bug.")
        elif n_live < expect:
            dead = [c for c, v in live.items() if v == 0]
            print(f"  *** PARTIAL: these are all-zero: {dead}")
        else:
            print("  OK: every column fires at least once.")

        # each flag should fire on exactly one day
        multi = {c: v for c, v in live.items() if v > 1}
        if multi:
            print(f"  NOTE: fire on >1 day (expected 1 each): {multi}")

    # --- where does each anchor land -----------------------------------------
    for anchor in ["N", "D"]:
        if anchor in win.columns:
            hits = win.loc[win[anchor] != 0, time_col]
            print(f"\n'{anchor}' day-0 in window: "
                  f"{[d.date().isoformat() for d in hits] or 'NONE'}")

    # --- days with no festive signal at all ----------------------------------
    fest = [c for c in (N_BLOCK + D_BLOCK) if c in win.columns]
    if fest:
        blank = win.loc[(win[fest] == 0).all(axis=1), time_col]
        print(f"\nDays in the window with NO festive flag: {len(blank)}/98")
        if len(blank):
            print(f"  first: {blank.min().date()}   last: {blank.max().date()}")
            sep = blank[blank.dt.month == 9]
            print(f"  of those, {len(sep)} fall in September "
                  f"({sep.min().date() if len(sep) else '-'} to "
                  f"{sep.max().date() if len(sep) else '-'})")
            print("  On these days the model has no festive signal and can only")
            print("  extrapolate the baseline from the encoder.")
    print()


# ------------------------------------------------------------------ compare
if len(sources) == 2:
    print("=" * 70)
    print("DO THE TWO ARTIFACTS AGREE?")
    print("=" * 70)
    a, b = list(sources.values())
    ca = set(a.columns) - {time_col}
    cb = set(b.columns) - {time_col}
    if ca != cb:
        print(f"*** Column sets differ. only in pkl: {sorted(ca-cb)[:8]} | "
              f"only in parquet: {sorted(cb-ca)[:8]} ***")
    else:
        wa = a[(a[time_col] >= FC_START) & (a[time_col] <= FC_END)].set_index(time_col).sort_index()
        wb = b[(b[time_col] >= FC_START) & (b[time_col] <= FC_END)].set_index(time_col).sort_index()
        cols = sorted(ca)
        same = np.allclose(wa[cols].to_numpy(float), wb[cols].to_numpy(float), equal_nan=True)
        print("Values identical in the forecast window:", same)
        if not same:
            print("*** The pickle inference uses differs from the source parquet.")
            print("    Rebuild shared_cov.pkl. ***")
